In [0]:
%sql
-- 1. Crear el contenedor principal para el proyecto Nexus
CREATE CATALOG IF NOT EXISTS nexus_project;
USE CATALOG nexus_project;

-- 2. Crear el esquema (base de datos) para integridad de tuberías
CREATE SCHEMA IF NOT EXISTS pipe_integrity
COMMENT 'Datos para el modelo de Risk Scoring de Tuberías';

USE SCHEMA pipe_integrity;

In [0]:
%sql
-- Crear un volumen para guardar archivos crudos
CREATE VOLUME IF NOT EXISTS nexus_project.default.raw_data;

In [0]:
# 1. Definir ruta del dataset
csv_path = "/Volumes/nexus_project/default/raw_data/market_pipe_thickness_loss_dataset.csv"

# 2. Leer el dataset con Spark
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

# 3. Mostrar los primeros resultados del dataset
print("Dataset de Tuberías cargado exitosamente:")
display(df.limit(10))

In [0]:
# 1. Definir la ruta
csv_path = "/Volumes/nexus_project/default/raw_data/market_pipe_thickness_loss_dataset.csv"

# 2. Leer el archivo con PySpark
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

# 3. Crear la base de datos (esquema) y guardar la tabla
spark.sql("CREATE DATABASE IF NOT EXISTS nexus_integrity")
df_bronze.write.mode("overwrite").format("delta").saveAsTable("pipe_integrity.bronze_pipe_inspections")

print("✅ Capa BRONZE creada exitosamente en la base de datos 'pipe_integrity'")

In [0]:
from pyspark.sql.functions import col, md5, concat, lit, current_timestamp

# 1. Leer de la tabla Bronze
df_bronze = spark.read.table("pipe_integrity.bronze_pipe_inspections")

# 2. Transformación y Normalización (Capa Silver)
df_silver = df_bronze.select(
    # Creamos un ID único para cada tubería (Asset ID)
    md5(concat(col("Material"), col("Grade"), col("Pipe_Size_mm").cast("string"))).alias("pipe_id"),
    
    # Estandarizamos los nombres a minúsculas y sin espacios (Snake Case)
    col("Pipe_Size_mm").alias("diameter_mm"),
    col("Thickness_mm").alias("nominal_thickness_mm"),
    col("Material").alias("material"),
    col("Grade").alias("material_grade"),
    col("Max_Pressure_psi").alias("operating_pressure_psi"),
    col("Temperature_C").alias("operating_temp_c"),
    col("Corrosion_Impact_Percent").alias("corrosion_impact_pct"),
    col("Thickness_Loss_mm").alias("thickness_loss_mm"),
    col("Material_Loss_Percent").alias("material_loss_pct"),
    col("Time_Years").alias("years_in_service"),
    col("Condition").alias("inspection_status"),
    
    # Metadato de auditoría
    current_timestamp().alias("processed_at")
).distinct()

# 3. Guardar en la Capa Silver
# Mantenemos el esquema 'nexus_integrity'
df_silver.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pipe_integrity.silver_pipe_data")

print("✅ Capa SILVER creada con éxito en 'pipe_integrity.silver_pipe_data'")
display(df_silver.limit(5))

In [0]:
from pyspark.sql.functions import col, when, round, max as _max, min as _min

# 1. Leer de la capa Silver
df_silver = spark.read.table("pipe_integrity.silver_pipe_data")

# 2. Obtener valores máximos para normalizar Presión y Edad (Escala 0-100)
stats = df_silver.select(
    _max("operating_pressure_psi").alias("max_psi"),
    _max("years_in_service").alias("max_years")
).collect()[0]

max_psi = stats['max_psi']
max_years = stats['max_years']

# 3. Cálculo del Risk Score
df_gold = df_silver.withColumn(
    "risk_score",
    round(
        (col("corrosion_impact_pct") * 0.30) +
        (col("material_loss_pct") * 0.40) +
        ((col("operating_pressure_psi") / max_psi) * 100 * 0.15) +
        ((col("years_in_service") / max_years) * 100 * 0.15),
        2
    )
)

# 4. Categorización y Recomendaciones
df_gold = df_gold.withColumn(
    "risk_category",
    when(col("risk_score") >= 70, "CRÍTICO")
    .when(col("risk_score") >= 40, "ALTO")
    .when(col("risk_score") >= 20, "MEDIO")
    .otherwise("BAJO")
).withColumn(
    "recommended_action",
    when(col("risk_category") == "CRÍTICO", "Inspección Inmediata / Reparación")
    .when(col("risk_category") == "ALTO", "Programar mantenimiento en < 30 días")
    .when(col("risk_category") == "MEDIO", "Monitoreo trimestral")
    .otherwise("Inspección rutinaria anual")
)

# 5. Guardar en la Capa Gold (Tabla Final de Negocio)
df_gold.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pipe_integrity.gold_pipe_risk_analysis")

print("🔥 Capa GOLD generada. El modelo de Risk Scoring está listo para Dashboard.")
display(df_gold.select("pipe_id", "material", "risk_score", "risk_category", "recommended_action") \
       .orderBy(col("risk_score").desc()))

In [0]:
%sql
-- Mostrar las tablas del esquema
USE CATALOG nexus_project; 
USE SCHEMA pipe_integrity;       
SHOW TABLES;
